In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor, LGBMClassifier
import sys
import re
from sklearn.preprocessing import LabelEncoder

In [2]:
df_listings = pd.read_csv("../data/Listing/listed_transactions_enriched.csv")
df_listings.head()

C:\Users\mayab\AppData\Local\Temp\ipykernel_25244\2995931910.py:1: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df_listings = pd.read_csv("../data/Listing/listed_transactions_enriched.csv")


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,Year,year_month,rate_30yr_fixed
0,90000.0,1075010398,miriamlara03@gmail.com,NaN,NaN,Miriam,Lara,34.097939,-117.909653,1045 N Azusa 61,...,Covina Valley Unified,91722,NaN,0.0,NaN,NaN,1045 N Azusa 61,2024,2024-01,6.6425
1,1500000.0,1074974457,janelle@judsonre.com,NaN,NaN,Janelle,Judson,33.121241,-117.081614,NaN,...,NaN,92025,NaN,0.0,NaN,NaN,NaN,2024,2024-01,6.6425
2,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,NaN,90067,NaN,2105.0,177861.0,NaN,2220 Avenue Of The Stars 2704,2024,2024-01,6.6425
3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,Capistrano Unified,92677,NaN,254.0,5300.0,NaN,16 Palisades,2024,2024-01,6.6425
4,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,2024,2024-01,6.6425


In [3]:
df_sold = pd.read_csv("../data/Sold/sold_transactions_enriched.csv")
df_sold.head()

C:\Users\mayab\AppData\Local\Temp\ipykernel_25244\236767917.py:1: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sold = pd.read_csv("../data/Sold/sold_transactions_enriched.csv")


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled,Year,year_month,rate_30yr_fixed
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN,2024,2024-01,6.6425
1,HighDesert,HighDesert,NaN,NaN,NaN,NaN,NaN,0.0,535486633,eabrown@lee-associates.com,...,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024,2024-01,6.6425
2,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,75000.0,529986282,Joe@9WINWIN.com,...,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024,2024-01,6.6425
3,InlandValleys,InlandValleys,NaN,True,NaN,NaN,NaN,199000.0,529618166,carolthefinder@yahoo.com,...,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024,2024-01,6.6425
4,SouthwestRiversideCounty,SouthwestRiversideCounty,NaN,True,NaN,NaN,NaN,19500.0,522614340,jtavisola@tavisola.com,...,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024,2024-01,6.6425


##### Number of Rows and Columns

In [4]:
df_listings.shape

(853754, 87)

In [5]:
df_sold.shape

(591115, 87)

For the listing transactions, there are 853754 rows and 87 columns. With the sold transactions, there 591115 rows and 87 columns.

##### Review column data types

In [6]:
df_listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 853754 entries, 0 to 853753
Data columns (total 87 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   OriginalListPrice             850444 non-null  float64
 1   ListingKey                    853754 non-null  int64  
 2   ListAgentEmail                780379 non-null  str    
 3   CloseDate                     255893 non-null  str    
 4   ClosePrice                    233059 non-null  float64
 5   ListAgentFirstName            848740 non-null  str    
 6   ListAgentLastName             853678 non-null  str    
 7   Latitude                      741807 non-null  float64
 8   Longitude                     742511 non-null  float64
 9   UnparsedAddress               851597 non-null  str    
 10  PropertyType                  853754 non-null  str    
 11  LivingArea                    747280 non-null  float64
 12  ListPrice                     851619 non-null  float64


In [7]:
df_sold.info()

<class 'pandas.DataFrame'>
RangeIndex: 591115 entries, 0 to 591114
Data columns (total 87 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   BuyerAgentAOR                 519608 non-null  str    
 1   ListAgentAOR                  523069 non-null  str    
 2   Flooring                      347464 non-null  str    
 3   ViewYN                        531925 non-null  object 
 4   WaterfrontYN                  322 non-null     object 
 5   BasementYN                    9738 non-null    object 
 6   PoolPrivateYN                 516977 non-null  object 
 7   OriginalListPrice             589404 non-null  float64
 8   ListingKey                    591115 non-null  int64  
 9   ListAgentEmail                547714 non-null  str    
 10  CloseDate                     591115 non-null  str    
 11  ClosePrice                    591108 non-null  float64
 12  ListAgentFirstName            587682 non-null  str    


#### **Missing Value Analysis**

##### Calculate missing counts and percentages per column

##### *Listing*

In [8]:
missing_listing_counts = df_listings.isnull().sum()
missing_listing_counts

OriginalListPrice                 3310
ListingKey                           0
ListAgentEmail                   73375
CloseDate                       597861
ClosePrice                      620695
                                 ...  
MiddleOrJuniorSchoolDistrict    853754
UnparsedAddress.1                 2157
Year                                 0
year_month                           0
rate_30yr_fixed                      0
Length: 87, dtype: int64

In [9]:
missing_listing_percent = (df_listings.isnull().mean()) * 100
missing_listing_percent

OriginalListPrice                 0.387700
ListingKey                        0.000000
ListAgentEmail                    8.594396
CloseDate                        70.027315
ClosePrice                       72.701856
                                   ...    
MiddleOrJuniorSchoolDistrict    100.000000
UnparsedAddress.1                 0.252649
Year                              0.000000
year_month                        0.000000
rate_30yr_fixed                   0.000000
Length: 87, dtype: float64

In [10]:
missing_summary = pd.DataFrame({
    "missing_listing_counts": missing_listing_counts,
    "missing_listing_percent": missing_listing_percent
})

In [11]:
missing_listing_summary = missing_summary.sort_values(by="missing_listing_percent", ascending=False)
print(missing_listing_summary)

                              missing_listing_counts  missing_listing_percent
CoveredSpaces                                 853754                    100.0
AboveGradeFinishedArea                        853754                    100.0
MiddleOrJuniorSchoolDistrict                  853754                    100.0
ElementarySchoolDistrict                      853754                    100.0
FireplacesTotal                               853754                    100.0
...                                              ...                      ...
MlsStatus                                          0                      0.0
ListingContractDate                                0                      0.0
Year                                               0                      0.0
year_month                                         0                      0.0
rate_30yr_fixed                                    0                      0.0

[87 rows x 2 columns]


In [12]:
missing_listing_summary = missing_listing_summary[missing_listing_summary["missing_listing_counts"] > 0]
print(missing_listing_summary)

                              missing_listing_counts  missing_listing_percent
CoveredSpaces                                 853754               100.000000
AboveGradeFinishedArea                        853754               100.000000
MiddleOrJuniorSchoolDistrict                  853754               100.000000
ElementarySchoolDistrict                      853754               100.000000
FireplacesTotal                               853754               100.000000
...                                              ...                      ...
PostalCode                                       208                 0.024363
ListAgentLastName                                 76                 0.008902
ListAgentLastName.1                               76                 0.008902
StateOrProvince                                   66                 0.007731
CountyOrParish                                     1                 0.000117

[74 rows x 2 columns]


In [13]:
missing_listing_summary["missing_listing_percent"] = missing_listing_summary["missing_listing_percent"].round(2)
print(missing_listing_summary)

                              missing_listing_counts  missing_listing_percent
CoveredSpaces                                 853754                   100.00
AboveGradeFinishedArea                        853754                   100.00
MiddleOrJuniorSchoolDistrict                  853754                   100.00
ElementarySchoolDistrict                      853754                   100.00
FireplacesTotal                               853754                   100.00
...                                              ...                      ...
PostalCode                                       208                     0.02
ListAgentLastName                                 76                     0.01
ListAgentLastName.1                               76                     0.01
StateOrProvince                                   66                     0.01
CountyOrParish                                     1                     0.00

[74 rows x 2 columns]


In [14]:
missing_above_90 = missing_listing_summary[missing_listing_summary['missing_listing_percent'] > 90]
print(missing_above_90)

                              missing_listing_counts  missing_listing_percent
CoveredSpaces                                 853754                   100.00
AboveGradeFinishedArea                        853754                   100.00
MiddleOrJuniorSchoolDistrict                  853754                   100.00
ElementarySchoolDistrict                      853754                   100.00
FireplacesTotal                               853754                   100.00
TaxYear                                       852886                    99.90
BelowGradeFinishedArea                        850334                    99.60
BusinessType                                  847779                    99.30
TaxAnnualAmount                               847661                    99.29
CoBuyerAgentFirstName                         834049                    97.69
BuilderName                                   824452                    96.57
LotSizeDimensions                             805053            

Dropping Variables:
- MiddleOrJuniorSchoolDistrict                  
- ElementarySchoolDistrict
- CoveredSpaces
- AboveGradeFinishedArea
- FireplacesTotal
- TaxYear
- BelowGradeFinishedArea
- BusinessType
- TaxAnnualAmount
- CoBuyerAgentFirstName
- BuilderName
- LotSizeDimensions
- ElementarySchool
- MiddleOrJuniorSchool

In [15]:
missing_above_70 = missing_listing_summary[missing_listing_summary['missing_listing_percent'] > 70]
print(missing_above_70)

                              missing_listing_counts  missing_listing_percent
CoveredSpaces                                 853754                   100.00
AboveGradeFinishedArea                        853754                   100.00
MiddleOrJuniorSchoolDistrict                  853754                   100.00
ElementarySchoolDistrict                      853754                   100.00
FireplacesTotal                               853754                   100.00
TaxYear                                       852886                    99.90
BelowGradeFinishedArea                        850334                    99.60
BusinessType                                  847779                    99.30
TaxAnnualAmount                               847661                    99.29
CoBuyerAgentFirstName                         834049                    97.69
BuilderName                                   824452                    96.57
LotSizeDimensions                             805053            

In [16]:
missing_above_70.shape

(31, 2)

In [17]:
missing_above_70

,missing_listing_counts,missing_listing_percent
CoveredSpaces,853754,100.00
AboveGradeFinishedArea,853754,100.00
MiddleOrJuniorSchoolDistrict,853754,100.00
ElementarySchoolDistrict,853754,100.00
FireplacesTotal,853754,100.00
TaxYear,852886,99.90
BelowGradeFinishedArea,850334,99.60
BusinessType,847779,99.30
TaxAnnualAmount,847661,99.29
CoBuyerAgentFirstName,834049,97.69


To perserve the variables, it would be sufficient if each variable contain less than 70% of missing values in order to use imputation to replace the missingness in the data. 

So, variables with 70% of missing variables will be removed from the dataset because through imputation with variables with a significant number of missingness, it may contribute to bias towards certain values than others. 

Keep these key variables (from above 70):
- ClosePrice

In [18]:
df_listings_clean = df_listings.drop(columns=['MiddleOrJuniorSchoolDistrict', 'ElementarySchoolDistrict', 'CoveredSpaces',
                                             'AboveGradeFinishedArea', 'FireplacesTotal', 'TaxYear', 
                                             'BelowGradeFinishedArea', 'BusinessType', 'TaxAnnualAmount',
                                             'CoBuyerAgentFirstName', 'BuilderName', 'LotSizeDimensions',
                                             'ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool',
                                             'BuyerAgencyCompensation', 'BuyerAgencyCompensationType', 'BuildingAreaTotal',
                                             'CoListAgentFirstName', 'CoListAgentLastName', 'CoListOfficeName', 'BuyerOfficeAOR',
                                             'BuyerOfficeName', 'BuyerOfficeName.1', 'AssociationFeeFrequency', 'BuyerAgentFirstName',
                                             'BuyerAgentMlsId', 'BuyerAgentLastName', 'CloseDate', 'CloseDate.1'])

In [19]:
df_listings_clean.shape

(853754, 57)

In [20]:
# Calculate percentage for all columns
missing_pct = (df_listings_clean.isna().sum() / len(df_listings_clean)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

ClosePrice                  72.701856
SubdivisionName             66.606306
PurchaseContractDate        56.868372
MainLevelBedrooms           53.761622
HighSchoolDistrict          39.757237
AttachedGarageYN            34.093193
AssociationFee              32.320200
Stories                     25.941899
Levels                      19.215371
GarageSpaces                18.987085
FireplaceYN                 14.354369
NewConstructionYN           13.437243
Latitude                    13.112325
Latitude.1                  13.112325
Longitude.1                 13.029866
Longitude                   13.029866
LivingArea                  12.471274
LivingArea.1                12.471274
PropertySubType             12.201407
BedroomsTotal               12.073853
MLSAreaMajor                11.293066
LotSizeAcres                 9.554392
LotSizeSquareFeet            9.237556
LotSizeArea                  9.111875
ListAgentEmail               8.594396
BathroomsTotalInteger        8.192992
YearBuilt   

In [21]:
num_cols_with_missing = df_listings_clean.isna().any().sum()

print(f"Number of variables with missing values: {num_cols_with_missing}")

Number of variables with missing values: 44


In [22]:
cols_dropping = missing_pct[(missing_pct < 1) & (missing_pct > 0)]
cols_dropping

OriginalListPrice           0.387700
ListAgentFirstName          0.587289
ListAgentLastName           0.008902
UnparsedAddress             0.252649
ListPrice                   0.250072
ListAgentFullName           0.026706
CountyOrParish              0.000117
ListAgentFirstName.1        0.587289
StreetNumberNumeric         0.441813
City                        0.112562
ContractStatusChangeDate    0.809484
ListPrice.1                 0.250072
StateOrProvince             0.007731
ListAgentLastName.1         0.008902
PostalCode                  0.024363
UnparsedAddress.1           0.252649
dtype: float64

In [23]:
cols_dropped = ['OriginalListPrice', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListPrice', 'ListAgentFullName', 'CountyOrParish',
                'ListAgentFirstName.1', 'StreetNumberNumeric', 'City', 'ContractStatusChangeDate', 'ListPrice.1', 'StateOrProvince', 'ListAgentLastName.1', 
                'PostalCode', 'UnparsedAddress.1']

In [24]:
df_listings_cleaned = df_listings_clean.dropna(subset=['OriginalListPrice', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListPrice', 'ListAgentFullName', 'CountyOrParish',
                'ListAgentFirstName.1', 'StreetNumberNumeric', 'City', 'ContractStatusChangeDate', 'ListPrice.1', 'StateOrProvince', 'ListAgentLastName.1', 
                'PostalCode', 'UnparsedAddress.1'])

In [25]:
df_listings_cleaned.shape

(833710, 57)

In [26]:
df_listings_clean.shape[0] - df_listings_cleaned.shape[0]

20044

In [27]:
# Calculate percentage for all columns
missing_pct = (df_listings_cleaned.isna().sum() / len(df_listings_cleaned)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

ClosePrice               72.441616
SubdivisionName          66.577227
PurchaseContractDate     56.422617
MainLevelBedrooms        52.741841
HighSchoolDistrict       39.248300
AttachedGarageYN         33.947896
AssociationFee           31.688237
Stories                  25.104533
GarageSpaces             18.731213
Levels                   18.266184
FireplaceYN              13.974763
Latitude                 13.002603
Latitude.1               13.002603
Longitude.1              12.921400
Longitude                12.921400
NewConstructionYN        12.685946
LivingArea               12.108767
LivingArea.1             12.108767
PropertySubType          12.070024
BedroomsTotal            11.681880
MLSAreaMajor             11.262549
LotSizeAcres              9.532571
LotSizeSquareFeet         9.223711
LotSizeArea               9.102566
ListAgentEmail            8.555973
BathroomsTotalInteger     8.044164
YearBuilt                 7.941251
ParkingTotal              5.959386
dtype: float64


In [28]:
# Create a reference mapping of Names -> Emails
# Drop any rows that have missing emails so the map is 'clean'
email_lookup = df_listings_cleaned.dropna(subset=['ListAgentEmail']).drop_duplicates(['ListAgentFirstName', 'ListAgentLastName'])

# Set the names as the index to make the 'lookup' possible
email_lookup = email_lookup.set_index(['ListAgentFirstName', 'ListAgentLastName'])['ListAgentEmail']

# Create a function to apply the logic
def fill_missing_emails(row):
    # Only try to fill if the current email is missing
    if pd.isna(row['ListAgentEmail']):
        # Look up the name in our reference map
        return email_lookup.get((row['ListAgentFirstName'], row['ListAgentLastName']), "None")
    return row['ListAgentEmail']

# Apply it to the dataframe
df_listings_cleaned['ListAgentEmail'] = df_listings_cleaned.apply(fill_missing_emails, axis=1)

In [29]:
df_listings_cleaned['ListAgentEmail'].isna().sum()

np.int64(0)

In [30]:
(df_listings_cleaned['ListAgentEmail'] == 'None').sum()

np.int64(3297)

From the code above, I noticed that there are no missing values for list agent's first and last names, so to determine their emails, I tried to look up any matches by full name to get the list agent's emails for the rows that are missing them. If there are no matches, those emails will be preserved by having a value of "None" (i.e. no email provided).

Source: https://medium.com/@adedokunjuliusayobami/handling-missing-values-with-light-gbm-4a222d8af31b

Light Gradient Boosting Machine (LightGBM) is a tree-based ensemble learning approach to enhance efficiency and scalability on high-dimensional feature set and large-scale dataset. It builds multiple decision trees sequentially to minimize loss and improve predictive performance. It hadnles missing values by learning the best split direction for missing entries at each node. We are basically training the model to predict the missing values by using other features as predictors, as would in a normal regression/classification problem. 

In [31]:
new_df = df_listings_cleaned.copy()
pred_df = df_listings_cleaned.copy()
new_df.head()

,OriginalListPrice,ListingKey,ListAgentEmail,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,Year,year_month,rate_30yr_fixed
0,90000.0,1075010398,miriamlara03@gmail.com,NaN,Miriam,Lara,34.097939,-117.909653,1045 N Azusa 61,ManufacturedInPark,...,NaN,2.0,Covina Valley Unified,91722,0.00,NaN,1045 N Azusa 61,2024,2024-01,6.6425
2,1340000.0,1074973329,haleh360@Gmail.com,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,Residential,...,False,NaN,NaN,90067,2105.00,177861.0,2220 Avenue Of The Stars 2704,2024,2024-01,6.6425
3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,Residential,...,False,3.0,Capistrano Unified,92677,254.00,5300.0,16 Palisades,2024,2024-01,6.6425
4,3150000.0,1074936537,anader@dppre.com,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,Residential,...,NaN,2.0,NaN,91108,NaN,9404.0,1615 Waverly Road,2024,2024-01,6.6425
5,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,Residential,...,False,4.0,Walnut Valley Unified,91765,295.95,58232.0,2250 Indian Creek Road,2024,2024-01,6.6425


In [32]:
categorical_columns = new_df.select_dtypes(include=['object']).columns
continuous_columns = new_df.select_dtypes(include=['int64', 'float64']).columns

C:\Users\mayab\AppData\Local\Temp\ipykernel_25244\1767474806.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = new_df.select_dtypes(include=['object']).columns


Determine the categorical variables with not too many unique values. 

In [33]:
high_card_cols = df_listings_cleaned[categorical_columns].nunique()
high_card_cols = high_card_cols[high_card_cols > 300]
print(high_card_cols)

ListAgentEmail              103779
ListAgentFirstName           19142
ListAgentLastName            41436
UnparsedAddress             698545
ListOfficeName               24082
ListAgentFullName            98168
MLSAreaMajor                  1150
ListAgentFirstName.1         19142
SubdivisionName              21646
ListingId                   833577
City                          1322
ContractStatusChangeDate       848
PurchaseContractDate           831
ListingContractDate            821
ListAgentLastName.1          41436
HighSchoolDistrict             456
PostalCode                    5133
UnparsedAddress.1           698545
dtype: int64


In [34]:
df_listings_cleaned[categorical_columns].isna().sum()

ListAgentEmail                   0
ListAgentFirstName               0
ListAgentLastName                0
UnparsedAddress                  0
PropertyType                     0
ListOfficeName                   0
ListAgentFullName                0
MLSAreaMajor                 93897
CountyOrParish                   0
PropertyType.1                   0
MlsStatus                        0
ListAgentFirstName.1             0
AttachedGarageYN            283027
PropertySubType             100629
SubdivisionName             555061
ListingId                        0
City                             0
ContractStatusChangeDate         0
PurchaseContractDate        470401
ListingContractDate              0
StateOrProvince                  0
FireplaceYN                 116509
Levels                      152287
ListAgentLastName.1              0
NewConstructionYN           105764
HighSchoolDistrict          327217
PostalCode                       0
UnparsedAddress.1                0
year_month          

ListAgentEmail, MLSAreaMajor, SubdivisionName, PurchaseContractDate, and HighSchoolDistrict have more than 300 unique categorical values, making them computationally tiring to impute with LightGBM. 

In [35]:
df_listings_cleaned.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'ClosePrice',
       'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude',
       'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice',
       'DaysOnMarket', 'ListOfficeName', 'ListAgentFullName',
       'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'PropertyType.1',
       'MlsStatus', 'ListAgentFirstName.1', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'YearBuilt',
       'DaysOnMarket.1', 'StreetNumberNumeric', 'LivingArea.1', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'Longitude.1', 'PurchaseContractDate',
       'ListingContractDate', 'Latitude.1', 'ListPrice.1', 'StateOrProvince',
       'FireplaceYN', 'Stories', 'Levels', 'ListAgentLastName.1',
       'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'Associatio

In [36]:
reduced_categorical_columns = ['AttachedGarageYN', 'PropertySubType', 'FireplaceYN', 'Levels',
                               'NewConstructionYN']

In [37]:
df_listings_cleaned['AttachedGarageYN']

0         False
2           NaN
3          True
4          True
5          True
          ...  
853749      NaN
853750     True
853751     True
853752      NaN
853753      NaN
Name: AttachedGarageYN, Length: 833710, dtype: object

In [38]:
df_listings_cleaned['PropertySubType']

0                           NaN
2                   Condominium
3         SingleFamilyResidence
4         SingleFamilyResidence
5         SingleFamilyResidence
                  ...          
853749                      NaN
853750    SingleFamilyResidence
853751    SingleFamilyResidence
853752    SingleFamilyResidence
853753                      NaN
Name: PropertySubType, Length: 833710, dtype: str

In [39]:
df_listings_cleaned['FireplaceYN']


0           NaN
2         False
3          True
4          True
5          True
          ...  
853749      NaN
853750     True
853751     True
853752    False
853753    False
Name: FireplaceYN, Length: 833710, dtype: object

In [40]:
df_listings_cleaned['Levels']

0                 One
2                 One
3                 Two
4                 Two
5         ThreeOrMore
             ...     
853749            NaN
853750            Two
853751            Two
853752            One
853753            NaN
Name: Levels, Length: 833710, dtype: str

In [41]:
df_listings_cleaned['NewConstructionYN']

0           NaN
2         False
3         False
4           NaN
5         False
          ...  
853749      NaN
853750      NaN
853751      NaN
853752      NaN
853753      NaN
Name: NewConstructionYN, Length: 833710, dtype: object

In [42]:
reduced_categorical_columns

['AttachedGarageYN',
 'PropertySubType',
 'FireplaceYN',
 'Levels',
 'NewConstructionYN']

In [43]:
df_listings_cleaned['AttachedGarageYN'].value_counts()

AttachedGarageYN
True     424282
False    126401
Name: count, dtype: int64

In [44]:
def convert_to_numeric( continuous_df, categorical_columns, categorical, pred_col ):
    # Iterate through the list of categorical columns
    for column in categorical_columns:
      # Check if the column is the one to be predicted and is categorical
      if column == pred_col and categorical == True:
        continue # Skip this column if it is the prediction column
      else:
        # Convert the categorical column to numeric using pd.factorize
        continuous_df[ column ] = pd.factorize( continuous_df[ column ] )[ 0 ]

    return continuous_df

In [45]:
#set LGBM model based on the Target
def set_model_type( categorical ):
  if categorical == True:
    # Predicting a categorical column
    return LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
  else:
    # Predicting a continuous column
    return LGBMRegressor(n_estimators=100, random_state=42, verbose=-1) # Returns LGBMRegressor

In [63]:
def impute_numerical_lgbm(df_imputed, columns, categorical_data, categorical_columns):
    for col in columns:

        # Drop if too many missing
        if df_imputed[col].isna().sum() > df_imputed.shape[0] * 0.8:
            df_imputed.drop(col, axis=1, inplace=True)
            continue

        missing_idx = df_imputed[df_imputed[col].isna()].index

        if len(missing_idx) == 0:
            continue

        # Work on a copy
        temp_df = df_imputed.copy()

        temp_df['is_nan'] = 0
        temp_df.loc[missing_idx, 'is_nan'] = 1

        # Convert to numeric
        temp_df = convert_to_numeric(
            temp_df, categorical_columns, categorical_data, col
        )

        train = temp_df[temp_df['is_nan'] == 0]
        test = temp_df[temp_df['is_nan'] == 1]

        X_train = train.drop([col, 'is_nan'], axis=1)
        y_train = train[col]

        X_test = test.drop([col, 'is_nan'], axis=1)

        model = set_model_type(categorical_data)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        df_imputed.loc[missing_idx, col] = y_pred

    return df_imputed 

In [72]:
def impute_categorical_lgbm(df, categorical_cols):
    df_imputed = df.copy()
    for col in categorical_cols:
        missing_mask = df_imputed[col].isna()

        if not missing_mask.any():
            continue

        y = df_imputed.loc[~missing_mask, col]

        le = LabelEncoder()
        y_encoded = le.fit_transform(y)
        
        X = df_imputed.drop(columns=[col]).copy()
        
        for c in X.select_dtypes(include=['object', 'category']).columns:
            X[c] = X[c].astype('category').cat.codes

        # Fill remaining NaNs (LightGBM can handle, but safer)
        X = X.fillna(-999)

        X_train = X.loc[~missing_mask]
        X_test = X.loc[missing_mask]

        clf = LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            class_weight='balanced',
            verbosity=-1
        )

        clf.fit(X_train, y_encoded)

        y_pred = clf.predict(X_test)
        df_imputed.loc[missing_mask, col] = le.inverse_transform(y_pred)

    return df_imputed

In [96]:
categorical_to_encode = [c for c in reduced_categorical_columns if pred_df[c].nunique() <= 50]

# This dataframe is now numeric (dummies) and contains NaN indicators
df_encoded = pd.get_dummies(
    pred_df, 
    columns=categorical_to_encode, 
    dummy_na=True,
    dtype=int)

df_encoded = df_encoded.select_dtypes(include=['int', 'float', 'bool', 'number'])

df_encoded.columns = [re.sub(r'[\[\]<>, ]+', '_', str(col)) for col in df_encoded.columns]

df_encoded.head()

# Use the encoding code later for modeling

,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,LivingArea,ListPrice,DaysOnMarket,ListingKeyNumeric,ParkingTotal,...,Levels_Two,Levels_Two_MultiSplit,Levels_Two_MultiSplit_One,Levels_Two_One,Levels_Two_ThreeOrMore,Levels_Two_ThreeOrMore_MultiSplit,Levels_nan,NewConstructionYN_False,NewConstructionYN_True,NewConstructionYN_nan
0,90000.0,1075010398,NaN,34.097939,-117.909653,960.0,90000.0,0,1075010398,4.0,...,0,0,0,0,0,0,0,0,0,1
2,1340000.0,1074973329,NaN,34.052207,-118.408445,1301.0,1340000.0,127,1074973329,0.0,...,0,0,0,0,0,0,0,1,0,0
3,2500000.0,1074954552,NaN,33.496363,-117.691677,2788.0,2500000.0,1,1074954552,3.0,...,1,0,0,0,0,0,0,1,0,0
4,3150000.0,1074936537,NaN,34.119345,-118.111254,3250.0,3150000.0,1,1074936537,2.0,...,1,0,0,0,0,0,0,0,0,1
5,3090000.0,1074917818,NaN,33.984057,-117.802819,7456.0,3090000.0,0,1074917818,4.0,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
#df_imputed = pred_df.copy()

In [ ]:
'''
df_imputed = impute_numerical_lgbm(
    df_imputed,
    continuous_columns,
    categorical_data=False,
    categorical_columns=categorical_columns
)
'''

In [ ]:
'''
df_imputed = impute_categorical_lgbm(
    df_imputed,
    categorical_to_encode
)
'''

C:\Users\mayab\AppData\Local\Temp\ipykernel_25244\2934487808.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in X.select_dtypes(include=['object', 'category']).columns:
C:\Users\mayab\AppData\Local\Temp\ipykernel_25244\2934487808.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/doc

In [107]:
num_df = df_listings_cleaned.select_dtypes(include=['int64', 'float64'])
num_df.isna().sum().sum()

np.int64(2820095)

In [ ]:
#num_df_imputed = df_imputed.select_dtypes(include=['int64', 'float64'])
#num_df_imputed.isna().sum().sum()

np.int64(0)

All the original 2820095 numerical values were imputed. 

In [ ]:
cat_df = df_listings_cleaned.select_dtypes(include=['object'])
cat_df.isna().sum().sum()

C:\Users\mayab\AppData\Local\Temp\ipykernel_25244\930664820.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_df = df_listings_cleaned.select_dtypes(include=['object'])


np.int64(2204792)

In [ ]:
#cat_df_imputed = df_imputed.select_dtypes(include=['object'])
#cat_df_imputed.isna().sum().sum()

C:\Users\mayab\AppData\Local\Temp\ipykernel_25244\22488813.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_df_imputed = df_imputed.select_dtypes(include=['object'])


np.int64(1446576)

From 2204792 original categorical values to 1446576.

In [111]:
cat_columns = ['MLSAreaMajor', 'PropertySubType', 'SubdivisionName', 'PurchaseContractDate', 'HighSchoolDistrict']

In [ ]:
#cat_df_imputed[cat_columns].nunique()

MLSAreaMajor             1150
PropertySubType            34
SubdivisionName         21646
PurchaseContractDate      831
HighSchoolDistrict        456
dtype: int64

In [ ]:
#df_imputed.to_csv("../data/Listing/imputed_listings_data.csv", index=False)

### **NEXT STEPS (For the rest of Week 3)**

- Handle the other missing values for the categorical variables
- Conduct the missing values and outlier cleaning for sold_transactions

#### **Outlier Cleaning**

In [ ]:
df_listings_clean.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'ClosePrice',
       'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude',
       'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice',
       'DaysOnMarket', 'ListOfficeName', 'ListAgentFullName',
       'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'PropertyType.1',
       'MlsStatus', 'ListAgentFirstName.1', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'YearBuilt',
       'DaysOnMarket.1', 'StreetNumberNumeric', 'LivingArea.1', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'Longitude.1', 'PurchaseContractDate',
       'ListingContractDate', 'Latitude.1', 'ListPrice.1', 'StateOrProvince',
       'FireplaceYN', 'Stories', 'Levels', 'ListAgentLastName.1',
       'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'Associatio

In [ ]:
neg_counts = (df_listings_clean.select_dtypes(include='number') < 0).sum()

In [ ]:
neg_counts[neg_counts > 0]

Latitude               7
Longitude         742263
DaysOnMarket          37
ParkingTotal         177
DaysOnMarket.1        37
Longitude.1       742263
Latitude.1             7
dtype: int64

In [ ]:
neg_DaysOnMarket = df_listings_clean[df_listings_clean['DaysOnMarket'] < 0]
neg_DaysOnMarket.head()

,OriginalListPrice,ListingKey,ListAgentEmail,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,Year,year_month,rate_30yr_fixed
223,799000.0,1063549350,ainsleyhughes@kw.com,799000.0,Ainsley,Hughes,34.425577,-119.291855,11905 Silver Spur Street,Residential,...,NaN,2.0,NaN,93023,59.0,5885.0,11905 Silver Spur Street,2024,2024-01,6.6425
228,899000.0,1063528331,absea@comcast.net,810000.0,A.B.,Priceman DRE 0126...,39.383806,-123.789254,31530 Emerald Drive,Residential,...,False,2.0,NaN,95437,NaN,34848.0,31530 Emerald Drive,2024,2024-01,6.6425
752,11500.0,1061271257,teresa.fuller@compass.com,10500.0,Teresa,Fuller,34.081125,-118.363403,536 N Edinburgh Avenue,ResidentialLease,...,NaN,0.0,NaN,90048,0.0,6612.0,536 N Edinburgh Avenue,2024,2024-01,6.6425
1119,1599000.0,1060153479,robert@anppros.com,1625000.0,Robert,Perez,34.244449,-118.265167,3929 El Moreno Street,Residential,...,NaN,2.0,NaN,91214,NaN,5249.0,3929 El Moreno Street,2024,2024-01,6.6425
4167,469999.0,1059512539,khoren9@yahoo.com,NaN,Khoren,Barutyan,34.201700,-118.460053,15015 Sherman Way 103,Residential,...,False,2.0,Los Angeles Unified,91405,390.0,15574.0,15015 Sherman Way 103,2024,2024-01,6.6425


In [ ]:
rows_DaysOnMarket = neg_DaysOnMarket['DaysOnMarket']
rows_DaysOnMarket.head()

223    -48
228    -58
752    -16
1119    -1
4167   -33
Name: DaysOnMarket, dtype: int64

There are negative values for DaysOnMarket, to which I have decided to remove those values. 

In [ ]:
neg_ParkingTotal = df_listings_clean[df_listings_clean['ParkingTotal'] < 0]
neg_ParkingTotal.head()

,OriginalListPrice,ListingKey,ListAgentEmail,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,Year,year_month,rate_30yr_fixed
8376,1100000.0,1058691297,amarprop@aol.com,1120000.0,Lino,Amarante,NaN,NaN,Eastwood Court,ResidentialIncome,...,NaN,1.0,San Jose Unified,95116,NaN,5663.0,Eastwood Court,2024,2024-01,6.6425
8443,2300000.0,1058688466,mabroukhamza093@gmail.com,NaN,Hamza,Mabrouk,NaN,NaN,725 Hyde Street,CommercialSale,...,NaN,NaN,San Francisco Unified,94109,0.0,1650.0,725 Hyde Street,2024,2024-01,6.6425
12223,1258000.0,1058416008,trisha@trishamotter.com,1600000.0,Trisha,Motter,NaN,NaN,7385 Forsum Road,Residential,...,False,2.0,Other,95138,NaN,5227.0,7385 Forsum Road,2024,2024-01,6.6425
26306,1588888.0,1054056032,maryoproperties@yahoo.com,1550000.0,Mary,O'neill,NaN,NaN,2965 Calle De Las Estrella,Residential,...,False,2.0,Other,95148,120.0,3920.0,2965 Calle De Las Estrella,2024,2024-01,6.6425
30751,1599000.0,1061806243,kroyer.re@gmail.com,1620000.0,Kathy,Royer,NaN,NaN,6088 Pietz Court,Residential,...,False,2.0,San Jose Unified,95123,NaN,6098.0,6088 Pietz Court,2024,2024-02,6.7760


In [ ]:
rows_ParkingTotal = neg_ParkingTotal['ParkingTotal']
rows_ParkingTotal.head()

8376      -2.0
8443      -5.0
12223     -3.0
26306     -1.0
30751   -139.0
Name: ParkingTotal, dtype: float64

In [ ]:
non_neg = (df_listings_clean['DaysOnMarket'] >= 0) & (df_listings_clean['ParkingTotal'] >= 0) & (df_listings_clean['DaysOnMarket.1'] >= 0)

In [ ]:
listings_clean = df_listings_clean[non_neg].copy()

In [ ]:
listings_clean.shape

(801183, 57)

In [ ]:
rows_dropped = len(df_listings_clean) - len(listings_clean)
rows_dropped

52571

In [ ]:
neg_counts = (listings_clean.select_dtypes(include='number') < 0).sum()

In [90]:
neg_counts[neg_counts > 0]

Latitude            4
Longitude      694719
Longitude.1    694719
Latitude.1          4
dtype: int64

##### *Sold*

In [28]:
missing_sold_counts = df_sold.isnull().sum()
missing_sold_counts

BuyerAgentAOR       71507
ListAgentAOR        68046
Flooring           243651
ViewYN              59190
WaterfrontYN       590793
                    ...  
latfilled          495322
lonfilled          495322
Year                    0
year_month              0
rate_30yr_fixed         0
Length: 87, dtype: int64

In [29]:
missing_sold_percent = (df_sold.isnull().mean()) * 100
missing_sold_percent

BuyerAgentAOR      12.096969
ListAgentAOR       11.511466
Flooring           41.218883
ViewYN             10.013280
WaterfrontYN       99.945527
                     ...    
latfilled          83.794524
lonfilled          83.794524
Year                0.000000
year_month          0.000000
rate_30yr_fixed     0.000000
Length: 87, dtype: float64

In [30]:
missing_summary = pd.DataFrame({
    "missing_sold_counts": missing_sold_counts,
    "missing_sold_percent": missing_sold_percent
})

In [31]:
missing_sold_summary = missing_summary.sort_values(by="missing_sold_percent", ascending=False)
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115                 100.0
AboveGradeFinishedArea                     591115                 100.0
MiddleOrJuniorSchoolDistrict               591115                 100.0
ElementarySchoolDistrict                   591115                 100.0
FireplacesTotal                            591115                 100.0
...                                           ...                   ...
MlsStatus                                       0                   0.0
StateOrProvince                                 0                   0.0
Year                                            0                   0.0
year_month                                      0                   0.0
rate_30yr_fixed                                 0                   0.0

[87 rows x 2 columns]


In [32]:
missing_sold_summary = missing_sold_summary[missing_sold_summary["missing_sold_counts"] > 0]
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115            100.000000
AboveGradeFinishedArea                     591115            100.000000
MiddleOrJuniorSchoolDistrict               591115            100.000000
ElementarySchoolDistrict                   591115            100.000000
FireplacesTotal                            591115            100.000000
...                                           ...                   ...
PostalCode                                    165              0.027913
ListAgentFullName                             161              0.027237
ListingContractDate                            83              0.014041
ListAgentLastName                              61              0.010319
ClosePrice                                      7              0.001184

[74 rows x 2 columns]


In [33]:
missing_sold_summary["missing_sold_percent"] = missing_sold_summary["missing_sold_percent"].round(2)
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115                100.00
AboveGradeFinishedArea                     591115                100.00
MiddleOrJuniorSchoolDistrict               591115                100.00
ElementarySchoolDistrict                   591115                100.00
FireplacesTotal                            591115                100.00
...                                           ...                   ...
PostalCode                                    165                  0.03
ListAgentFullName                             161                  0.03
ListingContractDate                            83                  0.01
ListAgentLastName                              61                  0.01
ClosePrice                                      7                  0.00

[74 rows x 2 columns]


In [34]:
missing_above_90 = missing_sold_summary[missing_sold_summary['missing_sold_percent'] > 90]
print(missing_above_90)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115                100.00
AboveGradeFinishedArea                     591115                100.00
MiddleOrJuniorSchoolDistrict               591115                100.00
ElementarySchoolDistrict                   591115                100.00
FireplacesTotal                            591115                100.00
WaterfrontYN                               590793                 99.95
TaxYear                                    590748                 99.94
BusinessType                               589483                 99.72
TaxAnnualAmount                            588608                 99.58
BelowGradeFinishedArea                     588525                 99.56
BasementYN                                 581377                 98.35
BuilderName                                568536                 96.18
LotSizeDimensions                          560102               

In [35]:
missing_above_70 = missing_sold_summary[missing_sold_summary['missing_sold_percent'] > 70]
print(missing_above_70)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115                100.00
AboveGradeFinishedArea                     591115                100.00
MiddleOrJuniorSchoolDistrict               591115                100.00
ElementarySchoolDistrict                   591115                100.00
FireplacesTotal                            591115                100.00
WaterfrontYN                               590793                 99.95
TaxYear                                    590748                 99.94
BusinessType                               589483                 99.72
TaxAnnualAmount                            588608                 99.58
BelowGradeFinishedArea                     588525                 99.56
BasementYN                                 581377                 98.35
BuilderName                                568536                 96.18
LotSizeDimensions                          560102               

In [36]:
missing_above_70

,missing_sold_counts,missing_sold_percent
CoveredSpaces,591115,100.00
AboveGradeFinishedArea,591115,100.00
MiddleOrJuniorSchoolDistrict,591115,100.00
ElementarySchoolDistrict,591115,100.00
FireplacesTotal,591115,100.00
WaterfrontYN,590793,99.95
TaxYear,590748,99.94
BusinessType,589483,99.72
TaxAnnualAmount,588608,99.58
BelowGradeFinishedArea,588525,99.56


In [37]:
df_sold_clean = df_sold.drop(columns=['MiddleOrJuniorSchoolDistrict', 'ElementarySchoolDistrict', 'CoveredSpaces', 
                                      'FireplacesTotal', 'AboveGradeFinishedArea', 'WaterfrontYN',
                                      'TaxYear', 'BusinessType', 'TaxAnnualAmount', 'BelowGradeFinishedArea',
                                      'BasementYN', 'BuilderName', 'LotSizeDimensions', 'CoBuyerAgentFirstName',
                                      'OriginatingSystemName', 'OriginatingSystemSubName', 'ElementarySchool', 
                                      'MiddleOrJuniorSchool', 'BuyerAgencyCompensationType', 'BuyerAgencyCompensation', 
                                      'BuildingAreaTotal', 'HighSchool', 'latfilled', 'lonfilled', 'CoListAgentFirstName', 
                                      'CoListAgentLastName', 'CoListOfficeName', 'AssociationFeeFrequency'])

In [38]:
df_sold_clean.shape

(591115, 59)

#### **Outlier Cleaning**